# Evaluate Stance Results

Load one stance run folder, inspect `document_assignments.csv`, and compute macro F1 and related classification metrics.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)

# Change this to any run folder you want to evaluate later.
RUN_DIR = Path("../data_out/semeval2016_task6_stance/GenAIStanceZeroShotCoT_100_semeval2016_task6_stance_gpt-5-mini_equal_default")

DOC_ASSIGNMENTS_PATH = RUN_DIR / "document_assignments.csv"
SUMMARY_PATH = RUN_DIR / "summary.json"

print("Run dir:", RUN_DIR.resolve())
print("Document assignments:", DOC_ASSIGNMENTS_PATH.exists())
print("Summary:", SUMMARY_PATH.exists())

Run dir: /Users/nityaakalra/Desktop/nyt_topic_modeling/topic_modeling_paper/data_out/semeval2016_task6_stance/GenAIStanceZeroShotCoT_100_semeval2016_task6_stance_gpt-5-mini_equal_default
Document assignments: True
Summary: True


In [3]:
if not DOC_ASSIGNMENTS_PATH.exists():
    raise FileNotFoundError(f"Missing document_assignments.csv at {DOC_ASSIGNMENTS_PATH}")

df = pd.read_csv(DOC_ASSIGNMENTS_PATH)
required_cols = {"doc_id", "original_stance", "predicted_stance"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"document_assignments.csv is missing required columns: {missing}")

print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()

Rows: 100
Columns: ['run', 'doc_index', 'doc_id', 'content', 'query', 'original_stance', 'reasoning', 'predicted_stance']


,run,doc_index,doc_id,content,query,original_stance,reasoning,predicted_stance
0,0,0,2521,Before I formed you in the #womb I knew you. #prolifegen #prolifeyouth #God #SemST,Legalization of Abortion,against,Against legalization. The tweet invokes the sanctity of fetal life with the line 'Before I formed you in the #womb I knew you' and uses explicit pro-life ha...,against
1,0,1,316,"What happened to this world! Attacks in France, Tunisia and Kuwait Kill Dozens!! Is this what you do in Ramadan!!!!? #shameless #SemST",Atheism,neutral,"Step 1: The target is 'Atheism'. Step 2: The content complains about deadly attacks and calls out perpetrators during Ramadan (#shameless), referencing reli...",neutral
2,0,2,405,They were all filled with the Holy Spirit and began to speak in foreign tongues as the Holy Spirit prompted them to speak #SemST,Atheism,against,"Step 1: The target is 'Atheism'. Step 2: The post affirms belief in the Holy Spirit and describes a supernatural event (speaking in tongues), signaling a re...",against
3,0,3,1434,Feminists tend to get upset & leave comedy clubs because humor is a form of intelligence they lack. #GamerGate #SemST,Feminist Movement,against,"Step 1 — Identify the target: the statement addresses “Feminists,” which maps directly to the Feminist Movement. Step 2 — Determine sentiment and intent: th...",against
4,0,4,1658,"The lack intellectual integrity in a group of Atheists is easy to spot: just count their ""FeministAtheists"" #GamerGate #SemST",Feminist Movement,against,"Step 1: The query target is the Feminist Movement; the post explicitly calls out ""FeministAtheists,"" a subgroup tied to feminist identity. Step 2: The conte...",against


In [5]:
# Filter to rows that have both a gold label and a prediction.
eval_df = df.copy()
eval_df["original_stance"] = eval_df["original_stance"].fillna("").astype(str).str.strip()
eval_df["predicted_stance"] = eval_df["predicted_stance"].fillna("").astype(str).str.strip()
eval_df = eval_df[(eval_df["original_stance"] != "") & (eval_df["predicted_stance"] != "")]

print("Rows usable for evaluation:", len(eval_df))
print("Gold labels:")
print(eval_df["original_stance"].value_counts(dropna=False))
print("\nPredicted labels:")
print(eval_df["predicted_stance"].value_counts(dropna=False))


Rows usable for evaluation: 100
Gold labels:
original_stance
against    46
neutral    33
pro        21
Name: count, dtype: int64

Predicted labels:
predicted_stance
against    38
neutral    32
pro        30
Name: count, dtype: int64


In [6]:
labels = sorted(set(eval_df["original_stance"]) | set(eval_df["predicted_stance"]))
y_true = eval_df["original_stance"]
y_pred = eval_df["predicted_stance"]

metrics_df = pd.DataFrame([
    {
        "n_examples": len(eval_df),
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "macro_f1": round(f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0), 4),
        "weighted_f1": round(f1_score(y_true, y_pred, average="weighted", labels=labels, zero_division=0), 4),
    }
])
metrics_df

,n_examples,accuracy,macro_f1,weighted_f1
0,100,0.8,0.7928,0.8038


In [9]:
# report = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
# report_df = pd.DataFrame(report).transpose()
# report_df

In [8]:
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"gold:{x}" for x in labels], columns=[f"pred:{x}" for x in labels])
cm_df

,pred:against,pred:neutral,pred:pro
gold:against,35,5,6
gold:neutral,2,26,5
gold:pro,1,1,19
